### Voll daneben – Lösung Schritt für Schritt

*37. Bundeswettbewerb Informatik (2018), 1. Runde, Aufgabe 3*

In Las Vegas veranstaltet Casinobesitzer Al Capone Junior ein neues Gewinnspiel. Eine Runde des Spiels läuft so ab: Jeder Teilnehmer zahlt einen Einsatz von 25 Dollar. Den Betrag setzt er auf eine „Glückszahl“, eine ganze Zahl im Bereich von 1 bis 1000. Nachdem alle Teilnehmer gesetzt haben, wählt Al zehn Zahlen, ebenfalls im Bereich von 1 bis 1000.

Anschließend werden die Gewinne ausgezahlt: Für jeden Teilnehmer wird diejenige von Als Zahlen bestimmt, die am nächsten an der Glückszahl des Teilnehmers liegt. Der Abstand dieser Zahl zu seiner Glückszahl ist der Gewinn des Teilnehmers.

Ein Beispiel: Bei einer Runde des Gewinnspiels waren unter anderem fünf alte Bekannte von Al dabei. Sie setzten auf diese Glückszahlen: Bugsy: 1, Bonnie: 15, Clyde: 100, Mickey: 200, Lucky: 300. Al wählte danach diese Zahlen: 1, 35, 117, 321, 448, 500, 678, 780, 802, 999. Damit erhielt Bugsy keinen Gewinn, Bonnie erhielt 14, Clyde 17, Mickey 83 und Lucky 21 Dollar.

Da Al seinen Reichtum mehren möchte, will er seine Zahlen so wählen, dass er möglichst wenig Gewinn auszahlen muss.

#### Aufgabe 3

Schreibe ein Programm, das die Glückszahlen aller Teilnehmer einer Runde einliest. Es soll dann für Al zehn Zahlen zwischen 1 und 1000 auswählen, die zu einer geringen Auszahlung führen.

Wende dein Programm auf die Beispiele an, die du auf den BWINF-Webseiten findest. Für diese Beispiele soll Al weniger auszahlen müssen, als die Teilnehmer eingesetzt haben.

#### Dateien (`beispieldaten/`)

- `beispiel1.txt` bis `beispiel3.txt`: je Zeile die Glückszahl eines Teilnehmers (1 bis 1000)
- `eigene_testdaten/`: das Beispiel der Lösungshinweise mit 12 Zahlen (`beispiel-skizzen.txt`, dort mit 5 statt 10 Zahlen von Al) und ein Beispiel, bei dem Al Verlust macht (`beispiel-verlust.txt`)

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Sortieren und Gruppen benachbarter Zahlen:** Auf dem Zahlenstrahl gehören Zahlen, die nah beieinander liegen, zur selben Zahl von Al. Nach dem Sortieren sind das Stücke der Liste, die direkt hintereinander stehen.
- **Median:** Die mittlere Zahl einer sortierten Gruppe. Sie ist der Punkt mit der kleinsten Summe der Abstände zur Gruppe (der Mittelwert ist es nicht).
- **Dynamische Programmierung:** Ein Problem in kleinere Teilprobleme zerlegen, deren Lösungen in einer Tabelle speichern und beim Berechnen größerer Teilprobleme wiederverwenden.
- **Präfixsummen:** Mit einer Liste der „Summen bis hierhin“ berechnet man die Summe eines beliebigen Stücks einer Liste mit einer Subtraktion.

**Nützliche Funktionen und Bibliotheken**
- `sorted(liste)` sortiert, `min(...)` und `abs(...)` für Abstände.
- `itertools.combinations(liste, k)` liefert alle Auswahlen von `k` Elementen. Wir nutzen es nur, um die Lösung bei kleinen Beispielen durch Ausprobieren zu überprüfen.

## Die Idee

Jede Glückszahl wird von der **nächsten** Zahl von Al „getroffen“, und der Abstand dazu kostet Al Geld. Al sucht also zehn Zahlen, sodass die Summe dieser Abstände möglichst klein ist.

Dazu drei Beobachtungen:

1. **Al wählt am besten Zahlen der Teilnehmer.** Liegt eine Zahl von Al zwischen zwei Glückszahlen, kann man sie so verschieben, dass sie auf einer Glückszahl liegt, und es wird nie teurer (die Treffer links und rechts verändern ihre Kosten gegenläufig, man schiebt zu der Seite mit den meisten Treffern). Es genügt also, aus den Zahlen der Teilnehmer auszuwählen.
2. **Die Teilnehmer zerfallen in Gruppen benachbarter Zahlen.** Sortiert man die Glückszahlen, dann werden die Teilnehmer, die von derselben Zahl von Al getroffen werden, immer ein zusammenhängendes Stück der sortierten Liste sein. Al sucht also eine Einteilung der sortierten Liste in `m = 10` Stücke.
3. **In jedem Stück wählt Al den Median.** Für ein Stück ist die beste einzelne Zahl die mittlere der sortierten Zahlen. Bei 1, 1, 1, 50 ist das eine 1 (Kosten 49), nicht der Mittelwert 13 (Kosten 12 + 12 + 12 + 37 = 73).

Die Aufgabe ist damit: **Zerlege die sortierte Liste so in 10 Stücke, dass die Summe der Kosten der Stücke (jeweils zum Median) minimal wird.** Es gibt aber sehr viele Möglichkeiten, die Schnittstellen zu setzen (bei 199 Zahlen und 9 Schnitten sind es über 10^15), probieren scheidet also aus. Wir schauen uns erst zwei einfache Einteilungen an und dann eine, die garantiert die beste findet.

## Schritt 1: Daten einlesen und Auszahlung berechnen

Wir lesen die Glückszahlen ein und sortieren sie. Die Funktion `auszahlung` berechnet für eine Auswahl von Als Zahlen die Summe der Abstände (jede Glückszahl zur nächsten Zahl von Al). Zur Probe das Beispiel aus der Aufgabe: Bugsy 1, Bonnie 15, Clyde 100, Mickey 200, Lucky 300 mit Als zehn Zahlen. Erwartet: 0, 14, 17, 83 und 21 Dollar.

In [ ]:
def lies_zahlen(dateiname):
    """Liest die Glückszahlen (eine pro Zeile) und gibt sie sortiert als Liste zurück."""
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(z) for z in datei.read().split()]
    return sorted(zahlen)


def abstand_zur_naechsten(zahl, al):
    """Abstand von 'zahl' zur nächstgelegenen Zahl von Al."""
    beste = abs(zahl - al[0])
    for a in al:
        if abs(zahl - a) < beste:
            beste = abs(zahl - a)
    return beste


def auszahlung(teilnehmer, al):
    """Summe aller Abstände zur jeweils nächsten Zahl von Al."""
    summe = 0
    for zahl in teilnehmer:
        summe = summe + abstand_zur_naechsten(zahl, al)
    return summe


al_beispiel = [1, 35, 117, 321, 448, 500, 678, 780, 802, 999]
for name, zahl in [('Bugsy', 1), ('Bonnie', 15), ('Clyde', 100), ('Mickey', 200), ('Lucky', 300)]:
    print(name, abstand_zur_naechsten(zahl, al_beispiel))
print('Summe:', auszahlung([1, 15, 100, 200, 300], al_beispiel))

for nummer in range(1, 4):
    zahlen = lies_zahlen('beispieldaten/beispiel' + str(nummer) + '.txt')
    print('beispiel' + str(nummer) + ':', len(zahlen), 'Teilnehmer, Einnahmen', 25 * len(zahlen), 'Dollar, kleinste/größte Zahl', zahlen[0], zahlen[-1])

Bugsy 0
Bonnie 14
Clyde 17
Mickey 83
Lucky 21
Summe: 135
beispiel1: 199 Teilnehmer, Einnahmen 4975 Dollar, kleinste/größte Zahl 5 995
beispiel2: 100 Teilnehmer, Einnahmen 2500 Dollar, kleinste/größte Zahl 11 996
beispiel3: 100 Teilnehmer, Einnahmen 2500 Dollar, kleinste/größte Zahl 20 1000


Die Einnahmen sind 25 Dollar mal Zahl der Teilnehmer: 4975 Dollar bei Beispiel 1 und je 2500 Dollar bei Beispiel 2 und 3. Al braucht eine Auszahlung, die **darunter** liegt.

## Schritt 2: Zwei einfache Einteilungen

**Einteilung A: gleich breite Bereiche.** Wir teilen den Zahlenraum 1 bis 1000 in 10 Bereiche der Breite 100 und setzen Als Zahlen in die Mitte (50, 150, …, 950). Das hängt gar nicht von den Teilnehmern ab und ist nach den Lösungshinweisen „zu einfach“. Wir nehmen es nur als Vergleich.

**Einteilung B: gleich viele Teilnehmer pro Gruppe.** Wir teilen die sortierte Liste in 10 Stücke, die (möglichst) gleich viele Zahlen enthalten, und nehmen von jedem Stück den Median. Da die Stücke nicht überall gleich lang sind, nehmen wir die Schnitte bei `n * g // 10` für `g = 0, 1, …, 10`.

Für den Median einer sortierten Teilliste von `i` bis `j` (beide eingeschlossen) nehmen wir die Zahl in der Mitte, `t[(i + j) // 2]`. Bei gerader Länge sind zwei mittlere Zahlen gleich gut.

In [ ]:
def waehle_gleich_breit(m):
    breite = 1000 // m
    return [breite * g + breite // 2 for g in range(m)]


def waehle_gleich_viele(t, m):
    n = len(t)
    al = []
    for g in range(m):
        von = n * g // m               # erste Zahl der Gruppe
        bis = n * (g + 1) // m - 1     # letzte Zahl der Gruppe
        al.append(t[(von + bis) // 2])
    return al


print('Beispiel   Einnahmen  A (gleich breit)  B (gleich viele)')
for nummer in range(1, 4):
    t = lies_zahlen('beispieldaten/beispiel' + str(nummer) + '.txt')
    a = auszahlung(t, waehle_gleich_breit(10))
    b = auszahlung(t, waehle_gleich_viele(t, 10))
    print('%8d %10d %17d %17d' % (nummer, 25 * len(t), a, b))

Beispiel   Einnahmen  A (gleich breit)  B (gleich viele)
       1       4975              4950              4950
       2       2500              2488              2426
       3       2500              2540              2520


Beispiel 1 ist ein Sonderfall: Die Teilnehmer haben alle Vielfachen von 5 gewählt, da liegen beide Einteilungen schon beim Optimum (4950). Bei den Beispielen 2 und 3 sind sie schlecht: Bei Beispiel 3 machen beide sogar Verlust (A zahlt 2540, B zahlt 2520 bei 2500 Einnahmen), bei Beispiel 2 gewinnt Al nur 12 (A) beziehungsweise 74 Dollar (B). Beide setzen die Schnitte, ohne auf die Lücken und Häufungen der Zahlen zu achten. Wir brauchen eine Einteilung, die die Schnitte **optimal** setzt.

## Schritt 3: Kosten eines Stücks

Wie teuer ist ein Stück `t[i..j]`, wenn Al dafür den Median nimmt? Wir rechnen es nicht jedes Mal mit einer Schleife aus, denn wir brauchen es sehr oft. Mit **Präfixsummen** geht es mit einer Subtraktion: `summe[k]` ist die Summe der ersten `k` Zahlen. Die Summe von `t[a..b]` ist dann `summe[b + 1] - summe[a]`.

Liegt der Median an Position `mitte`, dann gilt:

- Links vom Median (Positionen `i` bis `mitte - 1`) kostet jede Zahl `t[mitte] - Zahl`. Das sind `(mitte - i)` mal `t[mitte]` minus die Summe dieser Zahlen.
- Rechts davon (Positionen `mitte + 1` bis `j`) kostet jede Zahl `Zahl - t[mitte]`: die Summe dieser Zahlen minus `(j - mitte)` mal `t[mitte]`.

Wir bauen eine Tabelle `K` mit `K[i][j]` für alle `i <= j`.

In [ ]:
def baue_kosten_tabelle(t):
    """K[i][j] = Kosten, wenn die Teilnehmer i bis j (einschließlich) von ihrem Median getroffen werden."""
    n = len(t)
    summe = [0]                                  # summe[k] = t[0] + ... + t[k-1]
    for zahl in t:
        summe.append(summe[-1] + zahl)
    K = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i, n):
            mitte = (i + j) // 2
            links = t[mitte] * (mitte - i) - (summe[mitte] - summe[i])
            rechts = (summe[j + 1] - summe[mitte + 1]) - t[mitte] * (j - mitte)
            K[i][j] = links + rechts
    return K


# Probe an 1, 1, 1, 50 (Kosten 49) und am Beispiel der Lösungshinweise
print(baue_kosten_tabelle([1, 1, 1, 50])[0][3])
t_skizze = lies_zahlen('eigene_testdaten/beispiel-skizzen.txt')
K_skizze = baue_kosten_tabelle(t_skizze)
print(t_skizze)
print('Stück 14 16 18 19 20 (Positionen 2 bis 6):', K_skizze[2][6], '- direkt:', auszahlung([14, 16, 18, 19, 20], [18]))

49
[7, 10, 14, 16, 18, 19, 20, 24, 26, 29, 30, 32]
Stück 14 16 18 19 20 (Positionen 2 bis 6): 9 - direkt: 9


## Schritt 4: Die beste Einteilung mit dynamischer Programmierung

Teilproblem: `best[l][j]` sei die kleinste Auszahlung für die ersten `j` Teilnehmer (`t[0..j-1]`), wenn Al dafür genau `l` Zahlen setzt, also die ersten `j` Zahlen in `l` Stücke geteilt werden.

- Für `l = 0` geht es nur ohne Teilnehmer: `best[0][0] = 0`, alles andere unmöglich (unendlich).
- Für `l >= 1` überlegen wir, wo das **letzte** Stück beginnt, bei Position `i`. Es umfasst `t[i..j-1]` und kostet `K[i][j-1]`. Davor liegen die ersten `i` Teilnehmer in `l - 1` Stücken, kosten also `best[l-1][i]`. Wir probieren alle `i` und nehmen das kleinste:

    `best[l][j] = min über i von ( best[l-1][i] + K[i][j-1] )`

Zusätzlich merken wir uns in `anfang[l][j]` das beste `i`. Das Ergebnis `best[m][n]` ist die kleinste Auszahlung. Die Stücke liest man von hinten ab: Das letzte beginnt bei `anfang[m][n]`, das davor bei `anfang[m-1][...]` und so weiter. Von jedem Stück nehmen wir den Median.

**Warum stimmt das?** Ob die ersten `i` Teilnehmer optimal eingeteilt sind, hängt nicht davon ab, was danach kommt. Hätte man sie besser einteilen können, wäre auch das Ganze besser geworden. Deshalb genügt es, für jedes Teilproblem nur die beste Lösung zu speichern.

In [ ]:
UNENDLICH = 10 ** 9
ANZAHL_SCHRITTE = 0


def beste_einteilung(t, m):
    """Gibt (kleinste Auszahlung, Liste der Anfangspositionen der m Stücke) zurück. Voraussetzung: m <= len(t)."""
    global ANZAHL_SCHRITTE
    ANZAHL_SCHRITTE = 0
    n = len(t)
    K = baue_kosten_tabelle(t)
    best = [[UNENDLICH] * (n + 1) for _ in range(m + 1)]
    anfang = [[0] * (n + 1) for _ in range(m + 1)]
    best[0][0] = 0
    for l in range(1, m + 1):
        for j in range(l, n + 1):                  # j Teilnehmer in l Stücken: j >= l
            for i in range(l - 1, j):              # letztes Stück beginnt bei i
                ANZAHL_SCHRITTE += 1
                kosten = best[l - 1][i] + K[i][j - 1]
                if kosten < best[l][j]:
                    best[l][j] = kosten
                    anfang[l][j] = i
    # von hinten ablesen
    starts = []
    j = n
    for l in range(m, 0, -1):
        i = anfang[l][j]
        starts.append(i)
        j = i
    starts.reverse()
    return best[m][n], starts


def waehle_optimal(t, m):
    """Als Zahlen (aufsteigend sortiert, genau m verschiedene) für minimale Auszahlung."""
    n = len(t)
    if n <= m:
        al = sorted(set(t))                        # jeder Teilnehmer wird genau getroffen: Auszahlung 0
    else:
        kosten, starts = beste_einteilung(t, m)
        starts.append(n)                           # Ende des letzten Stücks
        al = []
        for g in range(m):
            von = starts[g]
            bis = starts[g + 1] - 1
            zahl = t[(von + bis) // 2]
            if zahl not in al:
                al.append(zahl)
    # Falls Zahlen doppelt vorkamen oder es weniger Teilnehmer als Zahlen gibt: mit freien Zahlen auffüllen
    zahl = 1
    while len(al) < m:
        if zahl not in al:
            al.append(zahl)
        zahl = zahl + 1
    return sorted(al)


t_skizze = lies_zahlen('eigene_testdaten/beispiel-skizzen.txt')
kosten, starts = beste_einteilung(t_skizze, 5)
print('Skizze mit 5 Zahlen: Auszahlung', kosten, 'Stücke beginnen bei', starts)
print('Als Zahlen:', waehle_optimal(t_skizze, 5), '-> Auszahlung', auszahlung(t_skizze, waehle_optimal(t_skizze, 5)))

Skizze mit 5 Zahlen: Auszahlung 12 Stücke beginnen bei [0, 2, 4, 7, 9]
Als Zahlen: [7, 14, 19, 24, 30] -> Auszahlung 12


Für das Beispiel der Lösungshinweise (12 Teilnehmer, 5 Zahlen für Al) ergibt sich die Auszahlung 12, wie dort angegeben.

## Schritt 5: Ausgabe und alle Beispiele

Die Ausgabe zeigt Als Zahlen, die Einnahmen (25 Dollar je Teilnehmer), die Auszahlung und den Gewinn. Lange Zahlenlisten kürzen wir.

In [ ]:
def zeige(name, dateiname, m=10):
    t = lies_zahlen(dateiname)
    al = waehle_optimal(t, m)
    ausz = auszahlung(t, al)
    einnahmen = 25 * len(t)
    print(name)
    print('  Teilnehmer:', len(t), '(größte Zahl', str(t[-1]) + ')')
    print('  Al wählt:', ' '.join(str(a) for a in al))
    print('  Einnahmen:', einnahmen, 'Dollar, Auszahlung:', ausz, 'Dollar, Gewinn für Al:', einnahmen - ausz, 'Dollar')
    return ausz


ERGEBNIS = {}
for nummer in range(1, 4):
    ERGEBNIS[nummer] = zeige('Beispiel ' + str(nummer), 'beispieldaten/beispiel' + str(nummer) + '.txt')
    print('  Rechenschritte der Tabelle:', ANZAHL_SCHRITTE)

Beispiel 1
  Teilnehmer: 199 (größte Zahl 995)
  Al wählt: 50 145 240 335 430 525 630 735 840 945
  Einnahmen: 4975 Dollar, Auszahlung: 4950 Dollar, Gewinn für Al: 25 Dollar
  Rechenschritte der Tabelle: 190165
Beispiel 2
  Teilnehmer: 100 (größte Zahl 996)
  Al wählt: 59 172 315 368 421 539 651 777 862 929
  Einnahmen: 2500 Dollar, Auszahlung: 1924 Dollar, Gewinn für Al: 576 Dollar
  Rechenschritte der Tabelle: 46120
Beispiel 3
  Teilnehmer: 100 (größte Zahl 1000)
  Al wählt: 100 240 340 440 520 580 660 720 860 960
  Einnahmen: 2500 Dollar, Auszahlung: 2160 Dollar, Gewinn für Al: 340 Dollar
  Rechenschritte der Tabelle: 46120


Al macht in allen drei Beispielen Gewinn: 25 Dollar bei Beispiel 1, 576 bei Beispiel 2 und 340 bei Beispiel 3. Das sind genau die Werte der Lösungshinweise (Auszahlungen 4950, 1924 und 2160). Die Auswahl der Zahlen kann bei mehreren gleich guten Möglichkeiten von den Hinweisen abweichen, die Auszahlung ist aber bei jeder optimalen Lösung gleich.

Bei Beispiel 1 (alle Vielfachen von 5 von 5 bis 995) bleibt kaum etwas zu gewinnen. Mehr als diese 25 Dollar sind hier **nicht** möglich, denn unser Verfahren findet das Optimum.

## Schritt 6: Kann Al auch Verlust machen?

Ja. Im Extra-Beispiel der Lösungshinweise haben die 21 Teilnehmer die Zahlen 1, 50, 99, …, 981 gewählt, jeweils im Abstand 49. Al kann nur 10 dieser Zahlen nehmen. Die übrigen 11 Zahlen haben mindestens Abstand 49 zur nächsten Zahl von Al, also muss er mindestens `11 · 49 = 539` Dollar auszahlen, bei Einnahmen von nur `21 · 25 = 525`. Unser Programm findet genau diese Auszahlung:

In [ ]:
verlust = zeige('Extra-Beispiel (Verlust)', 'eigene_testdaten/beispiel-verlust.txt')

Extra-Beispiel (Verlust)
  Teilnehmer: 21 (größte Zahl 981)
  Al wählt: 1 50 99 148 197 344 491 638 785 932
  Einnahmen: 525 Dollar, Auszahlung: 539 Dollar, Gewinn für Al: -14 Dollar


## Schritt 7: Probe durch Ausprobieren

Stimmt unser Verfahren wirklich immer? Bei kleinen Beispielen können wir alle Möglichkeiten für Als Zahlen durchprobieren (alle Auswahlen aus 1 bis 20, also auch Zahlen, die kein Teilnehmer gewählt hat) und mit dem Verfahren vergleichen. Mit `itertools.combinations` erhält man alle Auswahlen.

In [ ]:
import itertools
import random


def beste_durch_probieren(t, m, obergrenze):
    beste = None
    for al in itertools.combinations(range(1, obergrenze + 1), m):
        a = auszahlung(t, al)
        if beste is None or a < beste:
            beste = a
    return beste


random.seed(37)
abweichungen = 0
for versuch in range(60):
    n = random.randint(4, 9)
    m = random.randint(1, 3)
    t = sorted(random.randint(1, 20) for _ in range(n))
    if auszahlung(t, waehle_optimal(t, m)) != beste_durch_probieren(t, m, 20):
        abweichungen += 1
print('60 Zufallsbeispiele, Abweichungen von der besten Lösung:', abweichungen)

60 Zufallsbeispiele, Abweichungen von der besten Lösung: 0


## Für Neugierige

- **Laufzeit.** Die Tabelle hat `m · n` Einträge, und für jeden probieren wir bis zu `n` Möglichkeiten. Das sind etwa `m · n² / 2` Schritte. Bei 199 Teilnehmern und 10 Zahlen sind das rund 200 000 (die Schleife oben hat bei Beispiel 1 gezählt). Für sehr viele Teilnehmer wären die Tabellen zu groß. Da die Zahlen aber nur von 1 bis 1000 gehen, könnte man Teilnehmer mit gleicher Zahl zusammenfassen (mit einem Gewicht, wie oft sie vorkommt).
- **Die Hinweise** gehen den gleichen Weg mit dynamischer Programmierung, nur zerlegen sie anders: Sie legen die Zahlen von Al fest und berechnen die Kosten der Teilnehmer zwischen zwei Zahlen von Al. Wir legen stattdessen die Stücke fest und nehmen den Median. Das Ergebnis ist dasselbe.
- **Andere Verfahren.** PAM (Partitioning Around Medoids) beginnt mit zehn beliebigen Teilnehmerzahlen und tauscht so lange eine Zahl von Al gegen eine andere Teilnehmerzahl, bis keine Verbesserung mehr möglich ist. Das ist einfach zu programmieren, findet aber nicht immer das Optimum. Auch zufällige Wahl von Teilnehmerzahlen mit vielen Wiederholungen und Verschieben einzelner Zahlen liefert gute Ergebnisse. Unser Verfahren ist besser, weil es garantiert optimal ist.

## Zum Schluss: Voll daneben mit MiniZinc?

Wir haben geprüft, ob sich die Aufgabe mit MiniZinc lösen lässt. Das Modell liegt nahe: zehn aufsteigende Variablen `d[1..10]` mit Werten von 1 bis 1000, und minimiert wird die Summe über alle Teilnehmer vom kleinsten Abstand:

```minizinc
int: n; int: m;
array[1..n] of int: t;
array[1..m] of var 1..1000: d;
constraint forall(k in 1..m-1)(d[k] < d[k+1]);
var int: kosten = sum(i in 1..n)(min(k in 1..m)(abs(t[i]-d[k])));
solve minimize kosten;
```

Gemessen (jeweils mit Zeitlimit 20 Sekunden):

- **Gecode** findet bei keinem Beispiel das Optimum. Nach 20 Sekunden stehen Auszahlungen von 32 373 (Beispiel 1), 17 041 (Beispiel 2) und 9 758 (Beispiel 3) da. Die Optima sind 4 950, 1 924 und 2 160, und ein Beweis fehlt ohnehin.
- **Chuffed** liefert bei Beispiel 1 innerhalb von 40 Sekunden keine Ausgabe, bei Beispiel 2 eine Fehlermeldung.
- **CP-SAT** liefert bei Beispiel 1 auch nach mehreren Minuten nichts (der Prozess musste beendet werden), bei Beispiel 2 eine Fehlermeldung.

Hauptgrund: Das Modell enthält keine Struktur, die der Solver ausnutzen könnte. Jede der zehn Variablen hat 1000 Werte, und die Summe aus `min` und `abs` über 100 bis 199 Teilnehmer gibt ihm nur schwache Schranken. (Mit den Teilnehmerzahlen als Wertebereich oder einem zusätzlichen Gruppenmodell wären Verbesserungen denkbar, das haben wir nicht weiter verfolgt.)

Deshalb haben wir kein MiniZinc-Modell aufgenommen. Die dynamische Programmierung nutzt, was der Solver nicht kennt (benachbarte Teilnehmer bilden Gruppen, der Median ist je Gruppe optimal), und ist in Sekundenbruchteilen fertig.